# 🖊️ Spanish Handwritten Character Detector — YOLOv8
> **Stage 1 of 2-stage OCR pipeline**: Locate all individual handwritten characters (single-class, class 0 = *trazo*).

```
Image → [DETECTOR (this model)] → character crops → [CLASSIFIER (best_classifier.pt)] → text
```

---
## 📦 Install Dependencies

In [ ]:
!pip install 'ultralytics>=8.2' albumentations opencv-python-headless onnx onnxruntime pycocotools PyYAML -q

---
## ⚙️ D-0 — Environment Setup

In [ ]:
# ── D-0.1  ALL imports in a single cell ───────────────────────────────────
import torch
import cv2
import numpy as np
import pandas as pd
import json
import hashlib
import shutil
import random
import math
import os
import sys
import yaml
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Optional

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image, ImageDraw, ImageFont

from tqdm.auto import tqdm

import torchvision
import torchvision.transforms as transforms

import ultralytics
from ultralytics import YOLO

import albumentations as A

import onnx
import onnxruntime as ort

warnings.filterwarnings('ignore')
print('✅ All imports successful')

In [ ]:
# ── D-0.2  Path Configuration — Candidate Lists ────────────────────────────
INPUT_BASE = Path('/kaggle/input')
WORK       = Path('/kaggle/working')

OCR_CANDIDATE_ROOTS = [
    INPUT_BASE / 'datasets/danielperegrinoperez/spanish-ocr-dataset',
    INPUT_BASE / 'spanish-ocr-dataset',
    INPUT_BASE / 'danielperegrinoperez-spanish-ocr-dataset',
]
EMNIST_CANDIDATE_ROOTS = [
    INPUT_BASE / 'datasets/crawford/emnist',
    INPUT_BASE / 'emnist',
    INPUT_BASE / 'crawford-emnist',
]
VERACK_CANDIDATE_ROOTS = [
    INPUT_BASE / 'datasets/verack/spanish-handwritten-characterswords',
    INPUT_BASE / 'verack/spanish-handwritten-characterswords',
    INPUT_BASE / 'spanish-handwritten-characterswords',
]
CHAR_MAP_CANDIDATE_PATHS = [
    INPUT_BASE / 'datasets/danielperegrinoperez/char-map/char_map.json',
    INPUT_BASE / 'char-map/char_map.json',
    INPUT_BASE / 'danielperegrinoperez-char-map/char_map.json',
]

# ── Writable work directories ──────────────────────────────────────────────
BASE            = WORK / 'detector'
CROPS_DIR       = BASE / 'crops'
DATASET_DIR     = BASE / 'dataset'
RUNS_DIR        = BASE / 'runs'
EXPORTS_DIR     = BASE / 'exports'
VIS_DIR         = BASE / 'visualizations'
EMNIST_WORK_DIR = WORK / 'emnist_data'

for d in [
    CROPS_DIR / 'base', CROPS_DIR / 'emnist', CROPS_DIR / 'verack',
    DATASET_DIR / 'images' / 'train',
    DATASET_DIR / 'images' / 'val',
    DATASET_DIR / 'images' / 'test',
    DATASET_DIR / 'labels' / 'train',
    DATASET_DIR / 'labels' / 'val',
    DATASET_DIR / 'labels' / 'test',
    RUNS_DIR, EXPORTS_DIR,
    VIS_DIR / 'plana_validation',
    EMNIST_WORK_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print('✅ Directory tree created')

In [ ]:
# ── D-0.3  Hyperparameters ─────────────────────────────────────────────────
YOLO_MODEL    = 'yolov8n.pt'
YOLO_IMG_SIZE = 640
YOLO_EPOCHS   = 80
YOLO_BATCH    = 16
YOLO_PATIENCE = 15

COMP_CANVAS_SIZE    = 640
COMP_MIN_CHAR_H     = 32
COMP_MAX_CHAR_H     = 120
COMP_MIN_SPACING    = 5
COMP_MAX_OVERLAP_IOU = 0.05
COMP_TOTAL_TRAIN    = 8000
COMP_TOTAL_VAL      = 1000
COMP_TOTAL_TEST     = 1000
COMP_BATCH_SIZE     = 500

OCR_MAX_CROPS    = 30000
EMNIST_MAX_CROPS = 15000
VERACK_MAX_CROPS = 5000

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Composition type distribution
COMP_TYPES   = ['single', 'word', 'multiline', 'scattered', 'plana']
COMP_WEIGHTS = [0.10,     0.35,   0.25,        0.15,        0.15]

print('✅ Hyperparameters set')

In [ ]:
# ── D-0.4  GPU Verification ────────────────────────────────────────────────
assert torch.cuda.is_available(), '❌ CUDA not available — switch runtime to GPU!'
gpu = torch.cuda.get_device_properties(0)
print(f'✅ GPU  : {gpu.name}')
print(f'   VRAM : {gpu.total_memory / 1024**3:.1f} GB')
print(f'   CUDA : {torch.version.cuda}  |  cuDNN: {torch.backends.cudnn.version()}')
torch.backends.cudnn.benchmark = True

# ── D-0.5  Ultralytics Verification ───────────────────────────────────────
print(f'\n✅ Ultralytics: {ultralytics.__version__}')
_ = YOLO(YOLO_MODEL)
print('✅ YOLOv8n weights ready')

In [ ]:
# ── D-0.6  Dataset Discovery Diagnostic ───────────────────────────────────
def find_existing_path(candidates: List[Path]) -> Optional[Path]:
    """Return first existing path from candidates list, or None."""
    for p in candidates:
        if p.exists():
            return p
    return None

print('=' * 70)
print('📂 Scanning /kaggle/input/ for mounted datasets')
print('=' * 70)
if INPUT_BASE.exists():
    for item in sorted(INPUT_BASE.iterdir()):
        if item.is_dir():
            file_count = sum(1 for _ in item.rglob('*') if _.is_file())
            print(f'  📁 {item.name}/  ({file_count:,} files)')

ocr_root       = find_existing_path(OCR_CANDIDATE_ROOTS)
emnist_root_src = find_existing_path(EMNIST_CANDIDATE_ROOTS)
verack_root    = find_existing_path(VERACK_CANDIDATE_ROOTS)
char_map_path  = find_existing_path(CHAR_MAP_CANDIDATE_PATHS)

print(f'\n🎯 Base OCR dataset : {ocr_root or "❌ NOT FOUND"}')
print(f'🎯 EMNIST source    : {emnist_root_src or "❌ NOT FOUND (will download)"}')
print(f'🎯 verack source    : {verack_root or "❌ NOT FOUND"}')
print(f'🎯 char_map.json    : {char_map_path or "❌ NOT FOUND"}')

# Dynamic discovery fallback
if ocr_root is None:
    print('\n⚠️ OCR dataset not at expected paths. Scanning dynamically...')
    for item in INPUT_BASE.rglob('data.yaml'):
        print(f'  Found data.yaml: {item}')
    for item in INPUT_BASE.rglob('*.txt'):
        content = item.read_text().strip()
        if content and len(content.split()) == 5:
            print(f'  Found YOLO-style label dir: {item.parent}')
            break

---
## 🔍 D-1 — Dataset Inspection & Validation

In [ ]:
# ── D-1.0  Structure Discovery ─────────────────────────────────────────────
def discover_yolo_structure(root: Path) -> dict:
    """Discover actual YOLO dataset structure dynamically."""
    IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

    result = {
        'root': root,
        'images_dirs': [],
        'labels_dirs': [],
        'data_yaml': None,
        'splits_found': [],
        'total_images': 0,
        'total_labels': 0,
        'pairs': [],
    }

    for yaml_path in root.rglob('data.yaml'):
        result['data_yaml'] = yaml_path
        break

    # Strategy 1: images/train, images/val, images/test
    for subdir in root.rglob('images'):
        if subdir.is_dir():
            for split in ['train', 'val', 'test']:
                split_dir = subdir / split
                if split_dir.is_dir():
                    imgs = [f for f in split_dir.iterdir() if f.suffix.lower() in IMG_EXTS]
                    if imgs:
                        result['images_dirs'].append(split_dir)
                        result['splits_found'].append(split)
            # Flat: images/*.png
            imgs = [f for f in subdir.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS]
            if imgs and not result['images_dirs']:
                result['images_dirs'].append(subdir)
                result['splits_found'].append('flat')

    # Strategy 2: any dir with >100 images
    if not result['images_dirs']:
        for dirpath in root.rglob('*'):
            if dirpath.is_dir():
                imgs = [f for f in dirpath.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTS]
                if len(imgs) > 100:
                    result['images_dirs'].append(dirpath)
                    result['splits_found'].append('discovered')

    # Find label directories
    for img_dir in result['images_dirs']:
        lbl_dir = Path(str(img_dir).replace('/images', '/labels'))
        if lbl_dir.exists() and lbl_dir != img_dir:
            result['labels_dirs'].append(lbl_dir)
        else:
            lbl_dir = img_dir.parent / 'labels'
            if lbl_dir.exists():
                result['labels_dirs'].append(lbl_dir)

    # Build pairs
    for img_dir, lbl_dir in zip(result['images_dirs'], result['labels_dirs']):
        imgs = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
        lbls = {p.stem: p for p in lbl_dir.iterdir() if p.suffix.lower() == '.txt'}
        for stem in sorted(set(imgs) & set(lbls)):
            result['pairs'].append((imgs[stem], lbls[stem]))

    result['total_images'] = sum(
        len([f for f in d.iterdir() if f.suffix.lower() in IMG_EXTS])
        for d in result['images_dirs']
    )
    result['total_labels'] = sum(
        len([f for f in d.iterdir() if f.suffix.lower() == '.txt'])
        for d in result['labels_dirs']
    )

    return result

print('✅ discover_yolo_structure() defined')

In [ ]:
# ── D-1.1  Run Discovery and Report ───────────────────────────────────────
assert ocr_root is not None, "❌ Base OCR dataset not found! Add it via Kaggle 'Add Data'."

structure = discover_yolo_structure(ocr_root)
print(f"Root         : {structure['root']}")
print(f"data.yaml    : {structure['data_yaml']}")
print(f"Splits found : {structure['splits_found']}")
print(f"Image dirs   : {structure['images_dirs']}")
print(f"Label dirs   : {structure['labels_dirs']}")
print(f"Total images : {structure['total_images']:,}")
print(f"Total labels : {structure['total_labels']:,}")
print(f"Paired       : {len(structure['pairs']):,}")

all_pairs = structure['pairs']

In [ ]:
# ── D-1.2  Label Parsing and Statistics ───────────────────────────────────
label_stats = []
malformed   = []

for img_p, lbl_p in tqdm(all_pairs, desc='Parsing labels'):
    lines = lbl_p.read_text().strip().splitlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            malformed.append((str(lbl_p), line, 'wrong_fields'))
            continue
        try:
            cls = int(parts[0])
            cx, cy, w, h = [float(x) for x in parts[1:]]
        except ValueError:
            malformed.append((str(lbl_p), line, 'non_numeric'))
            continue
        label_stats.append({
            'img': str(img_p), 'lbl': str(lbl_p),
            'cx': cx, 'cy': cy, 'w': w, 'h': h
        })

df_lbl = pd.DataFrame(label_stats)
print(f'Valid bboxes : {len(df_lbl):,}  |  Malformed: {len(malformed):,}')

# ── Guard: handle empty results ────────────────────────────────────────────
if df_lbl.empty:
    print('⚠️ No bounding boxes found!')
    print('   Check diagnostic output above for correct paths.')
    boxes_per_img = pd.Series(dtype=int)
else:
    boxes_per_img = df_lbl.groupby('img').size()
    print('\nBoxes-per-image distribution:')
    print(boxes_per_img.value_counts().sort_index().head(10))
    print('\nBbox dimension stats:')
    print(df_lbl[['cx','cy','w','h']].describe())
    oob = df_lbl[(df_lbl[['cx','cy','w','h']] < 0).any(axis=1) | (df_lbl[['cx','cy','w','h']] > 1).any(axis=1)]
    print(f'Out-of-bounds bboxes: {len(oob):,}')

In [ ]:
# ── D-1.3  MD5 Deduplication ───────────────────────────────────────────────
def md5_file(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

hash2path = {}
kept, removed = [], []

# Prefer train split
ordered = sorted(all_pairs, key=lambda x: ('train' not in str(x[0]), str(x[0])))

for img_p, lbl_p in tqdm(ordered, desc='MD5 dedup'):
    h = md5_file(img_p)
    if h not in hash2path:
        hash2path[h] = (img_p, lbl_p)
        kept.append((img_p, lbl_p))
    else:
        removed.append((img_p, lbl_p))

print(f'Before dedup : {len(all_pairs):,}')
print(f'After dedup  : {len(kept):,}')
print(f'Duplicates   : {len(removed):,}')
all_pairs = kept

In [ ]:
# ── D-1.4  Image Quality Check ────────────────────────────────────────────
if len(all_pairs) == 0:
    print('⚠️ No pairs to check — skipping quality check')
else:
    sample_qc = random.sample(all_pairs, min(2000, len(all_pairs)))
    corrupt = []

    for img_p, lbl_p in tqdm(sample_qc, desc='Quality check'):
        img = cv2.imread(str(img_p), cv2.IMREAD_GRAYSCALE)
        if img is None or img.shape[0] == 0 or img.shape[1] == 0:
            corrupt.append((img_p, 'unreadable'))
            continue
        if img.std() < 5:
            corrupt.append((img_p, 'all_white'))
        elif img.mean() < 5:
            corrupt.append((img_p, 'all_black'))

    corrupt_set = {str(p) for p, _ in corrupt}
    all_pairs = [(i, l) for i, l in all_pairs if str(i) not in corrupt_set]
    print(f'Corrupted/degenerate found : {len(corrupt):,}')
    print(f'Remaining pairs            : {len(all_pairs):,}')

In [ ]:
# ── D-1.5  Label Sanitization ──────────────────────────────────────────────
if len(all_pairs) == 0:
    print('⚠️ No pairs to sanitize')
    clean_pairs = []
else:
    fixes = 0
    clean_pairs = []

    for img_p, lbl_p in tqdm(all_pairs, desc='Sanitizing labels'):
        lines = lbl_p.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                cls = int(parts[0])
                cx, cy, w, h = [float(x) for x in parts[1:]]
            except ValueError:
                continue
            if w < 0.01 or h < 0.01:
                fixes += 1
                continue
            orig = (cx, cy, w, h)
            cx = float(np.clip(cx, 0.001, 0.999))
            cy = float(np.clip(cy, 0.001, 0.999))
            w  = float(np.clip(w,  0.001, 0.999))
            h  = float(np.clip(h,  0.001, 0.999))
            if (cx, cy, w, h) != orig:
                fixes += 1
            new_lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
        if new_lines:
            clean_pairs.append((img_p, lbl_p, new_lines))

    print(f'Fixes applied  : {fixes:,}')
    print(f'Clean pairs    : {len(clean_pairs):,}')

In [ ]:
# ── D-1.6  Visualization ───────────────────────────────────────────────────
if not clean_pairs:
    print('⚠️ No clean pairs — skipping visualization')
else:
    sample_viz = random.sample(clean_pairs, min(25, len(clean_pairs)))
    fig, axes = plt.subplots(5, 5, figsize=(18, 18))
    for ax, (img_p, lbl_p, new_lines) in zip(axes.flatten(), sample_viz):
        img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        ax.imshow(img)
        for line in new_lines:
            _, cx, cy, bw, bh = map(float, line.split())
            x1 = (cx - bw/2) * W
            y1 = (cy - bh/2) * H
            rect = patches.Rectangle((x1, y1), bw*W, bh*H,
                                      linewidth=1.5, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
        ax.axis('off')
        ax.set_title(img_p.stem[:14], fontsize=7)
    plt.suptitle('D-1.6 — 25 random samples with GT bounding boxes', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(VIS_DIR / 'd1_sample_grid.png'), dpi=100, bbox_inches='tight')
    plt.show()

    # Histograms
    ws = [float(l.split()[3]) for _, _, ls in clean_pairs for l in ls]
    hs = [float(l.split()[4]) for _, _, ls in clean_pairs for l in ls]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(ws, bins=50, color='steelblue'); axes[0].set_title('BBox width')
    axes[1].hist(hs, bins=50, color='coral');    axes[1].set_title('BBox height')
    plt.tight_layout(); plt.show()

    # Image dimensions
    sample_dims = random.sample(clean_pairs, min(1000, len(clean_pairs)))
    img_dims = [cv2.imread(str(p)).shape[:2] for p,_,_ in tqdm(sample_dims, desc='Reading dims')]
    heights, widths = zip(*img_dims)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(widths,  bins=40, color='teal');   axes[0].set_title('Image width')
    axes[1].hist(heights, bins=40, color='salmon'); axes[1].set_title('Image height')
    plt.tight_layout(); plt.show()

---
## 🧩 D-2 — Multi-Character Image Composition
> **Most critical step.** Without multi-character training data, the detector fails on real inputs.

In [ ]:
# ── Crop-level augmentation pipeline ──────────────────────────────────────
CROP_AUG = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(5, 20), p=0.3),
    A.ElasticTransform(alpha=1, sigma=5, p=0.2),
    A.Rotate(limit=5, border_mode=cv2.BORDER_CONSTANT, value=255, p=0.4),
    A.Perspective(scale=(0.01, 0.03), p=0.2),
])

def augment_crop(crop_bgr: np.ndarray) -> np.ndarray:
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    out = CROP_AUG(image=rgb)['image']
    return cv2.cvtColor(out, cv2.COLOR_RGB2BGR)

print('✅ Crop augmentation pipeline ready')

In [ ]:
# ── D-2.1a  Crop Extraction — Base YOLO Dataset ────────────────────────────
def extract_base_crops(clean_pairs: list, max_crops: int = OCR_MAX_CROPS) -> List[Path]:
    crops = []
    selected = random.sample(clean_pairs, min(max_crops, len(clean_pairs)))
    for idx, (img_p, lbl_p, lines) in enumerate(tqdm(selected, desc='Extracting base crops')):
        img = cv2.imread(str(img_p))
        if img is None:
            continue
        H, W = img.shape[:2]
        for line in lines:
            _, cx, cy, bw, bh = map(float, line.split())
            pad = random.randint(2, 5)
            x1 = max(0, int((cx - bw/2) * W) - pad)
            y1 = max(0, int((cy - bh/2) * H) - pad)
            x2 = min(W, int((cx + bw/2) * W) + pad)
            y2 = min(H, int((cy + bh/2) * H) + pad)
            crop = img[y1:y2, x1:x2]
            if crop.shape[0] < 5 or crop.shape[1] < 5:
                continue
            crop_path = CROPS_DIR / 'base' / f'crop_{len(crops):06d}.png'
            cv2.imwrite(str(crop_path), crop)
            crops.append(crop_path)
            if len(crops) >= max_crops:
                break
        if len(crops) >= max_crops:
            break
    return crops

pool_base = extract_base_crops(clean_pairs)
print(f'✅ Base crops: {len(pool_base):,}')

In [ ]:
# ── D-2.1b  Crop Extraction — EMNIST ──────────────────────────────────────
def fix_emnist_orientation(arr: np.ndarray) -> np.ndarray:
    """Transpose + flip + invert to correct EMNIST orientation."""
    fixed = cv2.flip(cv2.transpose(arr), flipCode=1)
    return cv2.bitwise_not(fixed)

pool_emnist = []

try:
    # Copy .gz files to writable dir if needed
    if emnist_root_src is not None:
        for gz in emnist_root_src.rglob('*.gz'):
            dst = EMNIST_WORK_DIR / gz.name
            if not dst.exists():
                shutil.copy2(gz, dst)

    from torchvision.datasets import EMNIST
    em_ds = EMNIST(
        root=str(EMNIST_WORK_DIR), split='byclass',
        train=True, download=(emnist_root_src is None)
    )

    # Fast indexing via targets
    indices = list(range(len(em_ds)))
    random.shuffle(indices)
    indices = indices[:EMNIST_MAX_CROPS * 2]  # oversample, filter below

    for idx in tqdm(indices, desc='Extracting EMNIST crops'):
        if len(pool_emnist) >= EMNIST_MAX_CROPS:
            break
        pil_img, _ = em_ds[idx]
        arr = np.array(pil_img)
        arr = fix_emnist_orientation(arr)
        crop_bgr = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
        save_path = CROPS_DIR / 'emnist' / f'em_{len(pool_emnist):06d}.png'
        cv2.imwrite(str(save_path), crop_bgr)
        pool_emnist.append(save_path)

    print(f'✅ EMNIST crops: {len(pool_emnist):,}')
except Exception as e:
    print(f'⚠️  EMNIST skipped ({e}) — continuing with base crops only')

In [ ]:
# ── D-2.1c  Crop Extraction — verack ──────────────────────────────────────
pool_verack = []

if verack_root is not None:
    IMG_EXTS = {'.png', '.jpg', '.jpeg'}
    all_span = list(verack_root.rglob('*.png')) + list(verack_root.rglob('*.jpg'))
    random.shuffle(all_span)
    for img_p in tqdm(all_span, desc='verack crops'):
        if len(pool_verack) >= VERACK_MAX_CROPS:
            break
        img = cv2.imread(str(img_p))
        if img is None or img.shape[0] < 5 or img.shape[1] < 5:
            continue
        # Skip word images (wide aspect ratio)
        h, w = img.shape[:2]
        if w > 4 * h:
            continue
        save_path = CROPS_DIR / 'verack' / f'vk_{len(pool_verack):06d}.png'
        cv2.imwrite(str(save_path), img)
        pool_verack.append(save_path)
    print(f'✅ verack crops: {len(pool_verack):,}')
else:
    print('⚠️  verack dataset not found — skipping')

# ── D-2.2  Crop Pool Validation ────────────────────────────────────────────
IMG_EXTS_SET = {'.png', '.jpg', '.jpeg'}
ALL_CROPS = (
    [p for p in (CROPS_DIR / 'base').glob('*')   if p.suffix.lower() in IMG_EXTS_SET]
  + [p for p in (CROPS_DIR / 'emnist').glob('*') if p.suffix.lower() in IMG_EXTS_SET]
  + [p for p in (CROPS_DIR / 'verack').glob('*') if p.suffix.lower() in IMG_EXTS_SET]
)
random.shuffle(ALL_CROPS)
print(f'\n🎯 Total crop pool: {len(ALL_CROPS):,}')
assert len(ALL_CROPS) >= 1000, f'Insufficient crops: {len(ALL_CROPS)}'

# Validate sample
for crop_path in random.sample(ALL_CROPS, min(100, len(ALL_CROPS))):
    img = cv2.imread(str(crop_path))
    assert img is not None, f'Corrupt crop: {crop_path}'
    assert img.shape[0] >= 5 and img.shape[1] >= 5
print('✅ Crop pool validated')

CROP_SOURCES = {
    'base':   len([p for p in (CROPS_DIR / 'base').glob('*')   if p.suffix.lower() in IMG_EXTS_SET]),
    'emnist': len([p for p in (CROPS_DIR / 'emnist').glob('*') if p.suffix.lower() in IMG_EXTS_SET]),
    'verack': len([p for p in (CROPS_DIR / 'verack').glob('*') if p.suffix.lower() in IMG_EXTS_SET]),
}
print(f'   Sources: {CROP_SOURCES}')

In [ ]:
# ── D-2.3  Background Generation ──────────────────────────────────────────
def generate_background(size: int = COMP_CANVAS_SIZE) -> np.ndarray:
    """Generate varied backgrounds to improve robustness."""
    bg_type = random.choice(['white', 'noise', 'gradient', 'ruled', 'grid', 'aged'])
    canvas  = np.full((size, size, 3), 255, dtype=np.uint8)

    if bg_type == 'noise':
        canvas = np.random.randint(235, 256, (size, size, 3), dtype=np.uint8)

    elif bg_type == 'gradient':
        for y in range(size):
            val = int(240 + 15 * y / size)
            canvas[y, :] = val

    elif bg_type == 'ruled':
        spacing   = random.randint(25, 45)
        line_color = random.randint(180, 220)
        for y in range(spacing, size, spacing):
            thickness = random.choice([1, 1, 1, 2])
            cv2.line(canvas, (0, y), (size, y),
                     (line_color, line_color, line_color), thickness)

    elif bg_type == 'grid':
        spacing = random.randint(25, 45)
        color   = random.randint(190, 225)
        for i in range(spacing, size, spacing):
            cv2.line(canvas, (i, 0), (i, size), (color, color, color), 1)
            cv2.line(canvas, (0, i), (size, i), (color, color, color), 1)

    elif bg_type == 'aged':
        canvas[:, :, 0] = random.randint(225, 245)
        canvas[:, :, 1] = random.randint(230, 248)
        canvas[:, :, 2] = random.randint(235, 255)
        for _ in range(random.randint(0, 15)):
            cx_ = random.randint(0, size)
            cy_ = random.randint(0, size)
            r   = random.randint(3, 15)
            col = tuple(random.randint(200, 235) for _ in range(3))
            cv2.circle(canvas, (cx_, cy_), r, col, -1)

    return canvas

print('✅ Background generator ready')

In [ ]:
# ── D-2.4  Composition Engine ──────────────────────────────────────────────
def compute_iou(box1: Tuple, box2: Tuple) -> float:
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0


def load_resize_crop(crop_path: Path, min_h: int = COMP_MIN_CHAR_H, max_h: int = COMP_MAX_CHAR_H) -> np.ndarray:
    img = cv2.imread(str(crop_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    target_h = random.randint(min_h, max_h)
    scale    = target_h / max(h, 1)
    target_w = max(1, int(w * scale))
    return cv2.resize(img, (target_w, target_h), interpolation=cv2.INTER_AREA)


def place_crop(canvas: np.ndarray, crop: np.ndarray, x_off: int, y_off: int):
    """Blend crop onto canvas using dark-pixel mask. Returns bbox or None."""
    H, W = canvas.shape[:2]
    h, w = crop.shape[:2]
    if x_off < 0 or y_off < 0 or x_off+w > W or y_off+h > H:
        return None
    roi  = canvas[y_off:y_off+h, x_off:x_off+w]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    mask = (gray < 200).astype(np.uint8)
    for c in range(3):
        roi[:, :, c] = np.where(mask, crop[:, :, c], roi[:, :, c])
    canvas[y_off:y_off+h, x_off:x_off+w] = roi
    return (x_off, y_off, x_off+w, y_off+h)


def px_to_yolo(box: Tuple, canvas_size: int = COMP_CANVAS_SIZE) -> str:
    x1, y1, x2, y2 = box
    cx = (x1+x2)/2 / canvas_size
    cy = (y1+y2)/2 / canvas_size
    bw = (x2-x1) / canvas_size
    bh = (y2-y1) / canvas_size
    return f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}'


def no_overlap(new_box: Tuple, placed: List[Tuple], max_iou: float = COMP_MAX_OVERLAP_IOU) -> bool:
    return all(compute_iou(new_box, b) <= max_iou for b in placed)


def compose_image(
    crop_pool: List[Path],
    canvas_size: int = COMP_CANVAS_SIZE,
    composition_type: str = 'word'
) -> Tuple[np.ndarray, List[str]]:
    """Generate a composite image with YOLO labels."""
    canvas = generate_background(canvas_size)
    labels  = []
    placed  = []

    def try_place(crop_path, x_off, y_off):
        crop = load_resize_crop(crop_path)
        if crop is None: return False
        crop = augment_crop(crop)
        box  = place_crop(canvas, crop, x_off, y_off)
        if box and no_overlap(box, placed):
            placed.append(box)
            labels.append(px_to_yolo(box, canvas_size))
            return True
        return False

    S = canvas_size

    if composition_type == 'single':
        x_off = random.randint(S//4, S//2)
        y_off = random.randint(S//4, S//2)
        try_place(random.choice(crop_pool), x_off, y_off)

    elif composition_type == 'word':
        n_chars  = random.randint(3, 10)
        x_cursor = random.randint(10, 40)
        y_base   = random.randint(S//4, 3*S//4)
        for _ in range(n_chars):
            if x_cursor >= S - 20: break
            crop = load_resize_crop(random.choice(crop_pool))
            if crop is None: continue
            crop = augment_crop(crop)
            h, w = crop.shape[:2]
            y_off = max(5, min(S-h-5, y_base - h + random.randint(-5,5)))
            box = place_crop(canvas, crop, x_cursor, y_off)
            if box and no_overlap(box, placed):
                placed.append(box); labels.append(px_to_yolo(box, S))
            x_cursor += w + random.randint(COMP_MIN_SPACING, 15)

    elif composition_type == 'multiline':
        n_lines  = random.randint(2, 4)
        line_h   = random.randint(40, 70)
        y_start  = random.randint(20, 60)
        line_gap = random.randint(10, 25)
        for li in range(n_lines):
            y_base   = y_start + li*(line_h+line_gap)
            if y_base > S-10: break
            x_cursor = random.randint(5, 20)
            for _ in range(random.randint(3, 8)):
                if x_cursor >= S-20: break
                crop = load_resize_crop(random.choice(crop_pool), max_h=line_h)
                if crop is None: continue
                crop = augment_crop(crop)
                h, w = crop.shape[:2]
                y_off = max(5, min(S-h-5, y_base-h))
                box = place_crop(canvas, crop, x_cursor, y_off)
                if box and no_overlap(box, placed):
                    placed.append(box); labels.append(px_to_yolo(box, S))
                x_cursor += w + random.randint(COMP_MIN_SPACING, 12)

    elif composition_type == 'scattered':
        n_chars = random.randint(5, 15)
        for _ in range(n_chars * 3):  # attempts
            if len(placed) >= n_chars: break
            crop = load_resize_crop(random.choice(crop_pool))
            if crop is None: continue
            crop = augment_crop(crop)
            h, w = crop.shape[:2]
            x_off = random.randint(0, max(0, S-w))
            y_off = random.randint(0, max(0, S-h))
            box = place_crop(canvas, crop, x_off, y_off)
            if box and no_overlap(box, placed):
                placed.append(box); labels.append(px_to_yolo(box, S))

    elif composition_type == 'plana':
        cols = random.randint(4, 7)
        rows = random.randint(4, 7)
        cell_w = S // (cols+1)
        cell_h = S // (rows+1)
        for row in range(rows):
            for col in range(cols):
                crop = load_resize_crop(random.choice(crop_pool),
                                        min_h=max(COMP_MIN_CHAR_H, cell_h-10),
                                        max_h=min(COMP_MAX_CHAR_H, cell_h+5))
                if crop is None: continue
                crop = augment_crop(crop)
                h, w = crop.shape[:2]
                x_off = (col+1)*cell_w - w//2 + random.randint(-4, 4)
                y_off = (row+1)*cell_h - h//2 + random.randint(-4, 4)
                box = place_crop(canvas, crop, max(0,x_off), max(0,y_off))
                if box and no_overlap(box, placed):
                    placed.append(box); labels.append(px_to_yolo(box, S))

    return canvas, labels

print('✅ Composition engine ready')

In [ ]:
# ── D-2.5  Composition Execution ───────────────────────────────────────────
def generate_composites(
    crop_pool: List[Path],
    split: str,
    total: int,
    comp_types: List[str] = COMP_TYPES,
    comp_weights: List[float] = COMP_WEIGHTS,
) -> int:
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split
    generated = 0
    buf = []

    pbar = tqdm(total=total, desc=f'Composing [{split}]')
    attempts = 0
    while generated < total:
        attempts += 1
        if attempts > total * 10:
            print(f'⚠️ Too many attempts for {split}, stopping at {generated}')
            break
        ctype  = random.choices(comp_types, weights=comp_weights, k=1)[0]
        canvas, labels = compose_image(ALL_CROPS, composition_type=ctype)
        if not labels:
            continue
        if ctype != 'single' and len(labels) < 2:
            continue
        stem = f'comp_{split}_{generated:07d}'
        buf.append((stem, canvas, labels))
        generated += 1
        pbar.update(1)
        if len(buf) >= COMP_BATCH_SIZE:
            for s, c, ll in buf:
                cv2.imwrite(str(img_dir / f'{s}.png'), c)
                (lbl_dir / f'{s}.txt').write_text('\n'.join(ll))
            buf = []
    for s, c, ll in buf:
        cv2.imwrite(str(img_dir / f'{s}.png'), c)
        (lbl_dir / f'{s}.txt').write_text('\n'.join(ll))
    pbar.close()
    return generated

n_train = generate_composites(ALL_CROPS, 'train', COMP_TOTAL_TRAIN)
n_val   = generate_composites(ALL_CROPS, 'val',   COMP_TOTAL_VAL)
n_test  = generate_composites(ALL_CROPS, 'test',  COMP_TOTAL_TEST)
print(f'✅ Composites — train: {n_train}  val: {n_val}  test: {n_test}')

In [ ]:
# ── D-2.6  Include Original Single-Character Data ─────────────────────────
def copy_real_singles(clean_pairs, split, n):
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split
    selected = random.sample(clean_pairs, min(n, len(clean_pairs)))
    for i, (img_p, lbl_p, lines) in enumerate(tqdm(selected, desc=f'Real singles [{split}]')):
        dst_img = img_dir / f'real_{i:06d}{img_p.suffix}'
        dst_lbl = lbl_dir / f'real_{i:06d}.txt'
        shutil.copy2(img_p, dst_img)
        dst_lbl.write_text('\n'.join(lines))
    return len(selected)

r_train = copy_real_singles(clean_pairs, 'train', 2000)
r_val   = copy_real_singles(clean_pairs, 'val',    250)
r_test  = copy_real_singles(clean_pairs, 'test',   250)
print(f'✅ Real singles — train: {r_train}  val: {r_val}  test: {r_test}')

In [ ]:
# ── D-2.7  Composition Statistics and Visualization ────────────────────────
for split in ['train', 'val', 'test']:
    img_count = len(list((DATASET_DIR/'images'/split).glob('*.png')))
    lbl_count = len(list((DATASET_DIR/'labels'/split).glob('*.txt')))
    print(f'[{split}]  images={img_count:,}  labels={lbl_count:,}')

# Sample composite visualization
comp_imgs = list((DATASET_DIR/'images'/'train').glob('comp_*.png'))[:25]
if comp_imgs:
    fig, axes = plt.subplots(5, 5, figsize=(18, 18))
    for ax, img_p in zip(axes.flatten(), comp_imgs):
        img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
        lbl_p = DATASET_DIR/'labels'/'train'/(img_p.stem+'.txt')
        H, W  = img.shape[:2]
        ax.imshow(img)
        if lbl_p.exists():
            for line in lbl_p.read_text().strip().splitlines():
                _, cx, cy, bw, bh = map(float, line.split())
                x1 = (cx-bw/2)*W; y1 = (cy-bh/2)*H
                rect = patches.Rectangle((x1,y1), bw*W, bh*H,
                                          linewidth=1.5, edgecolor='red', facecolor='none')
                ax.add_patch(rect)
        ax.axis('off')
        n_boxes = len(lbl_p.read_text().strip().splitlines()) if lbl_p.exists() else 0
        ax.set_title(f'{n_boxes} chars', fontsize=8)
    plt.suptitle('D-2.7 — Synthetic composite samples with bboxes', fontsize=13)
    plt.tight_layout()
    plt.savefig(str(VIS_DIR/'d2_composite_samples.png'), dpi=100, bbox_inches='tight')
    plt.show()

    # Chars-per-image histogram
    counts = []
    for lbl_p in (DATASET_DIR/'labels'/'train').glob('comp_*.txt'):
        counts.append(len(lbl_p.read_text().strip().splitlines()))
    plt.figure(figsize=(8,4))
    plt.hist(counts, bins=30, color='steelblue')
    plt.title('Characters-per-image distribution (train composites)')
    plt.xlabel('# characters'); plt.ylabel('count')
    plt.tight_layout(); plt.show()

---
## 🏋️ D-3 — YOLOv8 Training

In [ ]:
# ── D-3.1  data.yaml ───────────────────────────────────────────────────────
data_yaml_content = {
    'path'  : str(DATASET_DIR),
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : 1,
    'names' : ['trazo'],
}
data_yaml_path = DATASET_DIR / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)
print(f'✅ data.yaml → {data_yaml_path}')
print(data_yaml_path.read_text())

In [ ]:
# ── D-3.2  Training ────────────────────────────────────────────────────────
train_start = datetime.now()
print(f'🕐 Training start: {train_start.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'   Estimated duration: ~{YOLO_EPOCHS * 0.5:.0f} min on T4')

model = YOLO(YOLO_MODEL)

results = model.train(
    data          = str(data_yaml_path),
    epochs        = YOLO_EPOCHS,
    imgsz         = YOLO_IMG_SIZE,
    batch         = YOLO_BATCH,
    patience      = YOLO_PATIENCE,
    # ── Augmentation (CRITICAL) ────────────────────────────────────────────
    flipud        = 0.0,      # DISABLED — vertical flip destroys text
    fliplr        = 0.0,      # DISABLED — horizontal flip destroys text
    mosaic        = 0.4,      # reduced — avoid cutting characters
    mixup         = 0.1,
    copy_paste    = 0.1,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.3,
    shear         = 2.0,
    perspective   = 0.0001,
    hsv_h         = 0.01,
    hsv_s         = 0.3,
    hsv_v         = 0.3,
    erasing       = 0.1,
    close_mosaic  = 10,
    # ── Optimizer ─────────────────────────────────────────────────────────
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    warmup_momentum = 0.8,
    # ── Other ─────────────────────────────────────────────────────────────
    project       = str(RUNS_DIR),
    name          = 'detector_v1',
    exist_ok      = True,
    save          = True,
    save_period   = 10,
    plots         = True,
    verbose       = True,
    workers       = 4,
    seed          = SEED,
    single_cls    = True,
)

train_end = datetime.now()
train_minutes = (train_end - train_start).total_seconds() / 60

# ── D-3.3  Training Monitoring ─────────────────────────────────────────────
best_model_path = RUNS_DIR / 'detector_v1' / 'weights' / 'best.pt'
print(f'\n✅ Training complete')
print(f'   Duration      : {train_minutes:.1f} min')
print(f'   Best weights  : {best_model_path}')

# Save training config
train_config = {
    'model': YOLO_MODEL, 'epochs': YOLO_EPOCHS, 'batch': YOLO_BATCH,
    'imgsz': YOLO_IMG_SIZE, 'patience': YOLO_PATIENCE,
    'train_start': train_start.isoformat(), 'train_minutes': round(train_minutes, 2),
    'best_model': str(best_model_path),
}
with open(EXPORTS_DIR / 'train_config.json', 'w') as f:
    json.dump(train_config, f, indent=2)
print('✅ train_config.json saved')

---
## 📊 D-4 — Evaluation Metrics

In [ ]:
# ── D-4.1  YOLO built-in validation ───────────────────────────────────────
eval_model = YOLO(str(best_model_path))
metrics = eval_model.val(
    data    = str(data_yaml_path),
    split   = 'test',
    imgsz   = YOLO_IMG_SIZE,
    batch   = YOLO_BATCH,
    conf    = 0.25,
    iou     = 0.5,
    plots   = True,
)

mAP50    = float(metrics.box.map50)
mAP5095  = float(metrics.box.map)
precision = float(metrics.box.mp)
recall    = float(metrics.box.mr)
f1        = 2 * precision * recall / (precision + recall + 1e-8)
speed_ms  = metrics.speed.get('inference', 0.0)

print(f'\n📈 mAP@0.50      : {mAP50:.4f}')
print(f'   mAP@0.50:0.95 : {mAP5095:.4f}')
print(f'   Precision     : {precision:.4f}')
print(f'   Recall        : {recall:.4f}')
print(f'   F1            : {f1:.4f}')
print(f'   Inference     : {speed_ms:.1f} ms/img')

In [ ]:
# ── D-4.2  Evaluation by Composition Type ─────────────────────────────────
type_results = defaultdict(lambda: {'tp': 0, 'fn': 0, 'fp': 0, 'total': 0})

test_img_dir = DATASET_DIR / 'images' / 'test'
test_lbl_dir = DATASET_DIR / 'labels' / 'test'

for img_p in tqdm(list(test_img_dir.glob('*.png'))[:500], desc='Per-type eval'):
    # Determine composition type from filename
    stem = img_p.stem
    if stem.startswith('real_'):
        ctype = 'single'
    else:
        # Run inference to determine type from box count
        ctype = 'comp'  # generic

    lbl_p = test_lbl_dir / (stem + '.txt')
    if not lbl_p.exists():
        continue

    gt_lines = lbl_p.read_text().strip().splitlines()
    n_gt = len(gt_lines)

    img = cv2.imread(str(img_p))
    preds = eval_model.predict(source=img, imgsz=YOLO_IMG_SIZE, conf=0.25, iou=0.45, verbose=False)
    n_pred = len(preds[0].boxes)

    tp = min(n_gt, n_pred)
    fn = max(0, n_gt - n_pred)
    fp = max(0, n_pred - n_gt)

    type_results[ctype]['tp'] += tp
    type_results[ctype]['fn'] += fn
    type_results[ctype]['fp'] += fp
    type_results[ctype]['total'] += 1

print('\nPer-type detection summary:')
for ctype, r in type_results.items():
    prec = r['tp'] / max(r['tp']+r['fp'], 1)
    rec  = r['tp'] / max(r['tp']+r['fn'], 1)
    print(f'  [{ctype:10s}]  images={r["total"]}  TP={r["tp"]}  FN={r["fn"]}  FP={r["fp"]}  P={prec:.2f}  R={rec:.2f}')

In [ ]:
# ── D-4.3  Detection Count Accuracy ───────────────────────────────────────
gt_counts, pred_counts = [], []

for img_p in tqdm(list(test_img_dir.glob('*.png'))[:500], desc='Count accuracy'):
    lbl_p = test_lbl_dir / (img_p.stem + '.txt')
    if not lbl_p.exists(): continue
    n_gt = len(lbl_p.read_text().strip().splitlines())
    img  = cv2.imread(str(img_p))
    preds = eval_model.predict(source=img, imgsz=YOLO_IMG_SIZE, conf=0.25, iou=0.45, verbose=False)
    n_pred = len(preds[0].boxes)
    gt_counts.append(n_gt)
    pred_counts.append(n_pred)

gt_arr   = np.array(gt_counts)
pred_arr = np.array(pred_counts)
mae      = float(np.mean(np.abs(gt_arr - pred_arr)))
exact_pct = float(np.mean(gt_arr == pred_arr) * 100)
avg_det   = float(np.mean(pred_arr))

print(f'Count MAE              : {mae:.2f} characters')
print(f'Exact count match      : {exact_pct:.1f}%')
print(f'Avg detections/image   : {avg_det:.1f}')

---
## 🖼️ D-5 — Visualization

In [ ]:
# ── D-5.1  GT vs Predictions Grid (4 rows × 5 cols) ──────────────────────
sample_imgs = random.sample(list(test_img_dir.glob('*.png')), min(20, len(list(test_img_dir.glob('*.png')))))

fig, axes = plt.subplots(4, 5, figsize=(22, 18))
for ax, img_p in zip(axes.flatten(), sample_imgs):
    img     = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
    H, W    = img.shape[:2]
    lbl_p   = test_lbl_dir / (img_p.stem + '.txt')
    preds   = eval_model.predict(source=cv2.imread(str(img_p)), imgsz=YOLO_IMG_SIZE,
                                  conf=0.25, iou=0.45, verbose=False)
    ax.imshow(img)
    # GT — green
    if lbl_p.exists():
        for line in lbl_p.read_text().strip().splitlines():
            _, cx, cy, bw, bh = map(float, line.split())
            x1 = (cx-bw/2)*W; y1 = (cy-bh/2)*H
            ax.add_patch(patches.Rectangle((x1,y1), bw*W, bh*H,
                         linewidth=1.5, edgecolor='lime', facecolor='none'))
    # Pred — red
    for box in preds[0].boxes:
        x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu())
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=1.5, edgecolor='red', facecolor='none'))
        ax.text(x1, y1-2, f'{conf:.2f}', color='red', fontsize=6)
    ax.axis('off')
plt.suptitle('D-5.1 — Green=GT  Red=Predicted', fontsize=13)
plt.tight_layout()
plt.savefig(str(VIS_DIR/'d5_gt_vs_pred.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── D-5.2  Confidence Distribution ────────────────────────────────────────
tp_confs, fp_confs = [], []

for img_p in list(test_img_dir.glob('*.png'))[:300]:
    lbl_p = test_lbl_dir / (img_p.stem + '.txt')
    if not lbl_p.exists(): continue
    gt_lines = lbl_p.read_text().strip().splitlines()
    H_img, W_img = cv2.imread(str(img_p)).shape[:2]
    gt_boxes = []
    for line in gt_lines:
        _, cx, cy, bw, bh = map(float, line.split())
        gt_boxes.append((int((cx-bw/2)*W_img), int((cy-bh/2)*H_img),
                         int((cx+bw/2)*W_img), int((cy+bh/2)*H_img)))
    preds = eval_model.predict(source=cv2.imread(str(img_p)), imgsz=YOLO_IMG_SIZE,
                                conf=0.25, iou=0.45, verbose=False)
    for box in preds[0].boxes:
        x1,y1,x2,y2 = [int(v) for v in box.xyxy[0].cpu().numpy()]
        conf = float(box.conf[0].cpu())
        is_tp = any(compute_iou((x1,y1,x2,y2), g) >= 0.5 for g in gt_boxes)
        (tp_confs if is_tp else fp_confs).append(conf)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(tp_confs, bins=30, color='green', alpha=0.7); axes[0].set_title('TP confidence')
axes[1].hist(fp_confs, bins=30, color='red',   alpha=0.7); axes[1].set_title('FP confidence')
plt.suptitle('D-5.2 — Confidence distribution')
plt.tight_layout()
plt.savefig(str(VIS_DIR/'d5_confidence_dist.png'), dpi=100)
plt.show()

In [ ]:
# ── D-5.3  Error Analysis ──────────────────────────────────────────────────
errors = []
for img_p in list(test_img_dir.glob('*.png'))[:500]:
    lbl_p = test_lbl_dir / (img_p.stem + '.txt')
    if not lbl_p.exists(): continue
    n_gt   = len(lbl_p.read_text().strip().splitlines())
    preds  = eval_model.predict(source=cv2.imread(str(img_p)), imgsz=YOLO_IMG_SIZE,
                                 conf=0.25, iou=0.45, verbose=False)
    n_pred = len(preds[0].boxes)
    missed = max(0, n_gt - n_pred)
    fps    = max(0, n_pred - n_gt)
    errors.append({'img': img_p, 'missed': missed, 'fps': fps})

# 10 worst missed
worst_missed = sorted(errors, key=lambda x: x['missed'], reverse=True)[:10]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, e in zip(axes.flatten(), worst_missed):
    img = cv2.cvtColor(cv2.imread(str(e['img'])), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(f'missed={e["missed"]}', fontsize=8); ax.axis('off')
plt.suptitle('D-5.3 — Worst missed detections')
plt.tight_layout()
plt.savefig(str(VIS_DIR/'d5_worst_missed.png'), dpi=100)
plt.show()

# 10 worst FP
worst_fp = sorted(errors, key=lambda x: x['fps'], reverse=True)[:10]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, e in zip(axes.flatten(), worst_fp):
    img = cv2.cvtColor(cv2.imread(str(e['img'])), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(f'FP={e["fps"]}', fontsize=8); ax.axis('off')
plt.suptitle('D-5.3 — Worst false positives')
plt.tight_layout()
plt.savefig(str(VIS_DIR/'d5_worst_fp.png'), dpi=100)
plt.show()

---
## 📤 D-6 — Export

In [ ]:
# ── D-6.1  PyTorch Export ─────────────────────────────────────────────────
best_src = RUNS_DIR / 'detector_v1' / 'weights' / 'best.pt'
best_dst = EXPORTS_DIR / 'best_detector.pt'
shutil.copy2(best_src, best_dst)
print(f'✅ Saved: {best_dst}')

# ── D-6.2  ONNX Export ────────────────────────────────────────────────────
exp_model = YOLO(str(best_dst))
onnx_out  = exp_model.export(
    format   = 'onnx',
    imgsz    = YOLO_IMG_SIZE,
    opset    = 17,
    simplify = True,
    dynamic  = True,
    half     = False,
)

# ── Mover/renombrar solo si es necesario ──────────────────────────────────
onnx_dst = EXPORTS_DIR / 'best_detector.onnx'
onnx_out_path = Path(onnx_out)

if onnx_out_path.resolve() != onnx_dst.resolve():
    shutil.copy2(onnx_out_path, onnx_dst)
    print(f'✅ Copied to: {onnx_dst}')
else:
    print(f'✅ Already at: {onnx_dst}')

# Validate ONNX
onnx_model = onnx.load(str(onnx_dst))
onnx.checker.check_model(onnx_model)
print('✅ ONNX model valid')

session = ort.InferenceSession(str(onnx_dst),
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
dummy   = np.random.randn(1, 3, 640, 640).astype(np.float32)
outputs = session.run(None, {session.get_inputs()[0].name: dummy})
print(f'✅ ONNX runtime OK  →  output shape: {outputs[0].shape}')

---
## 🔤 D-7 — Inference Function with Reading-Order Sorting

In [ ]:
def detect_characters(
    image: np.ndarray,
    model_path: str,
    conf_threshold: float = 0.25,
    iou_threshold: float  = 0.45,
    line_tolerance: float = 0.5,
) -> List[Dict]:
    """
    Detect all characters in an image and return bounding boxes in reading order.

    Args:
        image          : BGR numpy array (any size)
        model_path     : path to best_detector.pt
        conf_threshold : minimum confidence
        iou_threshold  : NMS IoU threshold
        line_tolerance : fraction of median char height for line grouping

    Returns:
        List of dicts sorted left→right, top→bottom:
        [{x1, y1, x2, y2, confidence, line}, ...]
    """
    mdl     = YOLO(model_path)
    results = mdl.predict(
        image, conf=conf_threshold, iou=iou_threshold,
        imgsz=YOLO_IMG_SIZE, verbose=False
    )

    detections = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu())
        detections.append({
            'x1': int(x1), 'y1': int(y1),
            'x2': int(x2), 'y2': int(y2),
            'confidence': round(conf, 4),
        })

    if not detections:
        return []

    # Reading-order sort
    heights  = [d['y2'] - d['y1'] for d in detections]
    median_h = sorted(heights)[len(heights) // 2]
    tol      = line_tolerance * median_h

    detections.sort(key=lambda d: (d['y1'] + d['y2']) / 2)

    lines       = []
    current_line = [detections[0]]
    current_y    = (detections[0]['y1'] + detections[0]['y2']) / 2

    for d in detections[1:]:
        y_center = (d['y1'] + d['y2']) / 2
        if abs(y_center - current_y) <= tol:
            current_line.append(d)
        else:
            lines.append(current_line)
            current_line = [d]
            current_y    = y_center
    lines.append(current_line)

    ordered = []
    for line_idx, line in enumerate(lines):
        line.sort(key=lambda d: d['x1'])
        for d in line:
            d['line'] = line_idx
            ordered.append(d)

    return ordered


print('✅ detect_characters() defined')

# Quick smoke-test
test_img_p = random.choice(list(test_img_dir.glob('*.png')))
test_dets  = detect_characters(cv2.imread(str(test_img_p)), str(best_dst))
print(f'Smoke-test: detected {len(test_dets)} characters in {test_img_p.name}')

---
## 📄 D-8 — Plana (Practice Sheet) Validation

In [ ]:
# ── D-8.1  Generate 5 synthetic planas ────────────────────────────────────
plana_dir = VIS_DIR / 'plana_validation'
planas    = []

for i in range(5):
    canvas, labels = compose_image(ALL_CROPS, composition_type='plana')
    path = plana_dir / f'plana_{i:02d}.png'
    lbl_path = plana_dir / f'plana_{i:02d}.txt'
    cv2.imwrite(str(path), canvas)
    lbl_path.write_text('\n'.join(labels))
    planas.append({'img': path, 'lbl': lbl_path, 'n_gt': len(labels)})

print(f'✅ Generated {len(planas)} synthetic planas')
for p in planas:
    print(f'   {p["img"].name}  GT chars: {p["n_gt"]}')

In [ ]:
# ── D-8.2  Run Detection on Planas ────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
plana_results = []

for ax, p in zip(axes, planas):
    img_bgr = cv2.imread(str(p['img']))
    dets    = detect_characters(img_bgr, str(best_dst), conf_threshold=0.25)
    n_det   = len(dets)
    n_gt    = p['n_gt']
    plana_results.append({'expected': n_gt, 'detected': n_det})

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_rgb.shape[:2]
    ax.imshow(img_rgb)

    # GT — green
    for line in p['lbl'].read_text().strip().splitlines():
        _, cx, cy, bw, bh = map(float, line.split())
        x1 = (cx-bw/2)*W; y1 = (cy-bh/2)*H
        ax.add_patch(patches.Rectangle((x1,y1), bw*W, bh*H,
                     linewidth=1.5, edgecolor='lime', facecolor='none'))
    # Pred — red
    for d in dets:
        ax.add_patch(patches.Rectangle((d['x1'],d['y1']),
                     d['x2']-d['x1'], d['y2']-d['y1'],
                     linewidth=1.5, edgecolor='red', facecolor='none'))

    ax.set_title(f'GT={n_gt}  Det={n_det}', fontsize=9)
    ax.axis('off')

plt.suptitle('D-8.2 — Plana validation (Green=GT, Red=Pred)', fontsize=12)
plt.tight_layout()
plt.savefig(str(plana_dir / 'plana_results.png'), dpi=100)
plt.show()

total_exp  = sum(r['expected'] for r in plana_results)
total_det  = sum(r['detected'] for r in plana_results)
det_rate   = total_det / max(total_exp, 1)
print(f'\nPlana summary: expected={total_exp}  detected={total_det}  rate={det_rate:.2f}')

In [ ]:
# ── D-8.3  End-to-End Pipeline Demo ───────────────────────────────────────
print('=' * 60)
print('🚀 End-to-End Pipeline Demo')
print('=' * 60)

demo_img_bgr = cv2.imread(str(planas[0]['img']))
detections   = detect_characters(demo_img_bgr, str(best_dst))

print(f'\nDetected {len(detections)} characters')
print('\nReading order output:')
for det in detections:
    crop = demo_img_bgr[det['y1']:det['y2'], det['x1']:det['x2']]
    print(f"  Line {det['line']}  bbox=({det['x1']},{det['y1']},{det['x2']},{det['y2']})"
          f"  conf={det['confidence']}  crop_size={crop.shape[:2]}")
    # Next step: classify(crop, 'best_classifier.pt')

print('\n✅ Pipeline demo complete')
print('   → Pass each crop to best_classifier.pt for character recognition')

---
## 📝 D-9 — Report and Artifacts

In [ ]:
# ── D-9  Save detector_metrics_report.json ─────────────────────────────────
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')

train_total = len(list((DATASET_DIR/'images'/'train').glob('*.png')))
val_total   = len(list((DATASET_DIR/'images'/'val').glob('*.png')))
test_total  = len(list((DATASET_DIR/'images'/'test').glob('*.png')))

comp_dist = Counter()
for lbl_p in (DATASET_DIR/'labels'/'train').glob('comp_*.txt'):
    n = len(lbl_p.read_text().strip().splitlines())
    if n == 1:   comp_dist['single'] += 1
    elif n <= 10: comp_dist['word'] += 1
    elif n <= 32: comp_dist['multiline'] += 1
    else:        comp_dist['plana'] += 1

report = {
    'run_id'              : run_id,
    'model'               : YOLO_MODEL.replace('.pt', '').upper(),
    'num_classes'         : 1,
    'class_names'         : ['trazo'],
    'img_size'            : YOLO_IMG_SIZE,
    'total_train_images'  : train_total,
    'total_val_images'    : val_total,
    'total_test_images'   : test_total,
    'composition_distribution': dict(comp_dist),
    'crop_pool_size'      : len(ALL_CROPS),
    'crop_sources'        : CROP_SOURCES,
    'best_epoch'          : int(getattr(results, 'best_epoch', 0)),
    'mAP50'               : round(mAP50,    4),
    'mAP50_95'            : round(mAP5095,  4),
    'precision'           : round(precision, 4),
    'recall'              : round(recall,    4),
    'f1'                  : round(f1,        4),
    'avg_detections_per_image' : round(avg_det,   2),
    'count_accuracy_pct'  : round(exact_pct, 2),
    'inference_ms_per_image': round(speed_ms, 2),
    'training_time_minutes': round(train_minutes, 2),
    'plana_validation'    : {
        'total_expected'  : total_exp,
        'total_detected'  : total_det,
        'detection_rate'  : round(det_rate, 4),
    },
}

report_path = EXPORTS_DIR / 'detector_metrics_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'✅ Report saved: {report_path}')
print(json.dumps(report, indent=2))

---
## ✅ Summary

| Step | Description | Key Change vs v1 |
|------|-------------|------------------|
| D-0  | Imports, candidate-list paths, hyperparams, GPU, discovery diagnostic | ✨ Candidate lists, full diagnostic scan |
| D-1  | Dynamic structure discovery, label parsing, dedup, sanitization | ✨ `discover_yolo_structure()`, empty-DataFrame guard |
| D-2  | Crop extraction (3 sources), backgrounds, 5-type composition engine | ✨ 5 types, `COMP_MAX_OVERLAP_IOU`, batch write |
| D-3  | YOLOv8n training with AdamW, `close_mosaic=10`, `single_cls=True` | ✨ Full config, `train_config.json` |
| D-4  | mAP + per-type eval + count accuracy | ✨ D-4.2, D-4.3 new |
| D-5  | GT vs Pred grid, confidence dist, error analysis | ✨ D-5.2, D-5.3 new |
| D-6  | PT + ONNX export with dynamic batch | ✨ `dynamic=True` |
| D-7  | `detect_characters()` with reading-order sort | ✨ New section |
| D-8  | 5 planas, GT overlay, end-to-end demo | ✨ New section |
| D-9  | `detector_metrics_report.json` | ✨ New section |

**Outputs:**
- `/kaggle/working/detector/exports/best_detector.pt`
- `/kaggle/working/detector/exports/best_detector.onnx`
- `/kaggle/working/detector/exports/detector_metrics_report.json`
- `/kaggle/working/detector/exports/train_config.json`
- `/kaggle/working/detector/visualizations/`